In [1]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    prefix = 'https://codeload.github.com'
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()
        if not (filename_lower.endswith('.md') or filename_lower.endswith('.mdx')):
            continue
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [2]:
my_data = read_repo_data('kubernetes', 'website')
print(f"Documents found: {len(my_data)}")
print(my_data[0])

Error processing website-main/archetypes/blog-post.md: while parsing a block mapping
  in "<unicode string>", line 2, column 1
did not find expected key
  in "<unicode string>", line 3, column 27
Error processing website-main/archetypes/concepts.md: while parsing a block mapping
  in "<unicode string>", line 2, column 1
did not find expected key
  in "<unicode string>", line 2, column 27
Error processing website-main/archetypes/default.md: while parsing a block mapping
  in "<unicode string>", line 2, column 1
did not find expected key
  in "<unicode string>", line 2, column 27
Error processing website-main/archetypes/tasks.md: while parsing a block mapping
  in "<unicode string>", line 2, column 1
did not find expected key
  in "<unicode string>", line 2, column 27
Error processing website-main/archetypes/tutorials.md: while parsing a block mapping
  in "<unicode string>", line 2, column 1
did not find expected key
  in "<unicode string>", line 2, column 27
Documents found: 7501
{'nam

In [3]:
# How many documents?
print(len(my_data))

# Look at the first few
for doc in my_data[:3]:
    print(doc.get('title', 'no title'))
    print(doc['filename'])
    print('---')

7501
no title
website-main/.github/ISSUE_TEMPLATE/bug-report.md
---
no title
website-main/.github/ISSUE_TEMPLATE/feature-request.md
---
no title
website-main/.github/ISSUE_TEMPLATE/support.md
---


In [4]:
# See what metadata fields are available
print(my_data[1].keys())

dict_keys(['name', 'about', 'labels', 'content', 'filename'])


In [5]:
# Find docs about a specific topic
k8s_pods = [doc for doc in my_data if 'pod' in doc.get('title', '').lower()]
print(f"Docs about pods: {len(k8s_pods)}")

Docs about pods: 547


In [6]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [7]:
kuber_chunks = []

for doc in my_data:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    kuber_chunks.extend(chunks)


In [8]:
import re
text = my_data[45]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())


In [9]:
def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections


In [11]:
kuber_chunks = []

for doc in my_data:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        kuber_chunks.append(section_doc)


In [14]:
for doc in my_data[:10]:
    print(doc['filename'], len(doc.get('content', '')))

website-main/.github/ISSUE_TEMPLATE/bug-report.md 510
website-main/.github/ISSUE_TEMPLATE/feature-request.md 560
website-main/.github/ISSUE_TEMPLATE/support.md 457
website-main/.github/PULL_REQUEST_TEMPLATE.md 1292
website-main/CONTRIBUTING.md 2615
website-main/README.md 9284
website-main/SECURITY.md 836
website-main/code-of-conduct.md 147
website-main/content/bn/README.md 10699
website-main/content/bn/_common-resources/index.md 0


In [15]:
my_data[0]

{'name': 'Bug Report',
 'about': 'Report a bug encountered with Kubernetes Website',
 'labels': ['kind/bug'],
 'content': '**This is a Bug Report**\n\n<!-- Thanks for filing an issue! Before submitting, please fill in the following information. -->\n<!-- See https://kubernetes.io/docs/contribute/start/ for guidance on writing an actionable issue description. -->\n\n<!--Required Information-->\n**Problem:**\n\n**Proposed Solution:**\n\n**Page to Update:**\nhttps://kubernetes.io/...\n\n<!--Optional Information (remove the comment tags around information you would like to include)-->\n<!--Kubernetes Version:-->\n\n<!--Additional Information:-->',
 'filename': 'website-main/.github/ISSUE_TEMPLATE/bug-report.md'}

In [17]:
kuber_docs = sorted(my_data, key=lambda doc: len(doc.get('content', '')))

In [26]:
kuber_copy = my_data[5].copy()
copy_content = kuber_copy.pop('content')
copy_content

"# The Kubernetes documentation\n\n[![Netlify Status](https://api.netlify.com/api/v1/badges/be93b718-a6df-402a-b4a4-855ba186c97d/deploy-status)](https://app.netlify.com/sites/kubernetes-io-main-staging/deploys) [![GitHub release](https://img.shields.io/github/release/kubernetes/website.svg)](https://github.com/kubernetes/website/releases/latest)\n\nThis repository contains the assets required to build the [Kubernetes website and documentation](https://kubernetes.io/). We're glad that you want to contribute!\n\n- [Contributing to the docs](#contributing-to-the-docs)\n- [Localization READMEs](#localization-readmes)\n\n## Using this repository\n\nYou can run the website locally using [Hugo (Extended version)](https://gohugo.io/), or you can run it in a container runtime. We strongly recommend using the container runtime, as it gives deployment consistency with the live website.\n\n## Prerequisites\n\nTo use this repository, you need the following installed locally:\n\n- [npm](https://www.

In [27]:
sliding_window(copy_content,2000,1000)

[{'start': 0,
  'chunk': "# The Kubernetes documentation\n\n[![Netlify Status](https://api.netlify.com/api/v1/badges/be93b718-a6df-402a-b4a4-855ba186c97d/deploy-status)](https://app.netlify.com/sites/kubernetes-io-main-staging/deploys) [![GitHub release](https://img.shields.io/github/release/kubernetes/website.svg)](https://github.com/kubernetes/website/releases/latest)\n\nThis repository contains the assets required to build the [Kubernetes website and documentation](https://kubernetes.io/). We're glad that you want to contribute!\n\n- [Contributing to the docs](#contributing-to-the-docs)\n- [Localization READMEs](#localization-readmes)\n\n## Using this repository\n\nYou can run the website locally using [Hugo (Extended version)](https://gohugo.io/), or you can run it in a container runtime. We strongly recommend using the container runtime, as it gives deployment consistency with the live website.\n\n## Prerequisites\n\nTo use this repository, you need the following installed locally

In [28]:
def hybrid_chunking(doc, min_size=100, max_size=2000, window_size=2000, step=1000):
    doc_copy = doc.copy()
    content = doc_copy.pop('content')
    
    sections = split_markdown_by_level(content, level=2)
    
    # if no sections were found, fall back to sliding window on the whole doc
    if not sections:
        chunks = sliding_window(content, window_size, step)
        result = []
        for chunk in chunks:
            c = doc_copy.copy()
            c['chunk'] = chunk['chunk']
            c['method'] = 'sliding_window'
            result.append(c)
        return result
    
    result = []
    pending = []  # accumulates sections that are too short

    for section in sections:
        if len(section) < min_size:
            # too short — hold it and merge with the next section
            pending.append(section)
        elif len(section) > max_size:
            # flush any pending short sections first
            if pending:
                merged = '\n\n'.join(pending)
                c = doc_copy.copy()
                c['chunk'] = merged
                c['method'] = 'merged_short_sections'
                result.append(c)
                pending = []
            # too long — apply sliding window
            chunks = sliding_window(section, window_size, step)
            for chunk in chunks:
                c = doc_copy.copy()
                c['chunk'] = chunk['chunk']
                c['method'] = 'sliding_window'
                result.append(c)
        else:
            # just right — merge any pending short sections with this one
            if pending:
                pending.append(section)
                merged = '\n\n'.join(pending)
                c = doc_copy.copy()
                c['chunk'] = merged
                c['method'] = 'section_merged'
                pending = []
            else:
                c = doc_copy.copy()
                c['chunk'] = section
                c['method'] = 'section'
            result.append(c)
    
    # flush any remaining pending sections
    if pending:
        merged = '\n\n'.join(pending)
        c = doc_copy.copy()
        c['chunk'] = merged
        c['method'] = 'merged_short_sections'
        result.append(c)
    
    return result

In [29]:
hybrid_chunking(my_data[5])

[{'filename': 'website-main/README.md',
  'chunk': '## Using this repository\n\nYou can run the website locally using [Hugo (Extended version)](https://gohugo.io/), or you can run it in a container runtime. We strongly recommend using the container runtime, as it gives deployment consistency with the live website.',
  'method': 'section'},
 {'filename': 'website-main/README.md',
  'chunk': '## Prerequisites\n\nTo use this repository, you need the following installed locally:\n\n- [npm](https://www.npmjs.com/)\n- [Go](https://go.dev/)\n- [Hugo (Extended version)](https://gohugo.io/)\n- A container runtime, like [Docker](https://www.docker.com/).\n\n> [!NOTE]\nMake sure to install the Hugo extended version specified by the `HUGO_VERSION` environment variable in the [`netlify.toml`](netlify.toml#L11) file.\n\nBefore you start, install the dependencies. Clone the repository and navigate to the directory:\n\n```bash\ngit clone https://github.com/kubernetes/website.git\ncd website\n```\n\nTh